# Tutorial 03: Noise Mitigation

Real quantum hardware has noise. QuoNic provides two error mitigation techniques.

## Adding Noise

Simulate noisy quantum circuits with depolarizing noise.

In [ ]:
from quonic import qgate, qshow, reset
from quonic.gates import H, CX

reset()
qgate(H, 0)
qgate(CX, 0, 1)
# No noise: clean Bell state
print("No noise:")
qshow()

In [ ]:
# With 5% depolarizing noise
print("With noise=0.05:")
qshow(noise=0.05)

## ZNE (Zero-Noise Extrapolation)

ZNE amplifies noise by folding the circuit, then extrapolates to zero noise.

In [ ]:
from quonic import zne
from quonic.ir import Circuit, GateOperation

# Build a simple circuit: X gate
c = Circuit()
c.add(GateOperation("x", (0,)))
c.add(GateOperation("measure", (0,)))

# ZNE with linear extrapolation
res_lin = zne(c, noise=0.05, target="1", shots=4096, extrapolation="linear")
print(f"Raw:           {res_lin.values[0]:.3f}")
print(f"ZNE linear:    {res_lin.extrapolated:.3f}")

# ZNE with exponential extrapolation
res_exp = zne(c, noise=0.05, target="1", shots=4096, extrapolation="exponential")
print(f"ZNE exponential: {res_exp.extrapolated:.3f}")
print(f"Ideal: 1.000")

## Readout Calibration

Correct measurement errors by inverting the confusion matrix.

In [ ]:
from quonic import calibrate
from quonic.noise import NoiseModel

n = 2
noise = NoiseModel(readout=0.05)

# Build calibration matrix
cal = calibrate(n, backend="native", shots=4096, noise=noise)

# Run a noisy Bell circuit
reset()
qgate(H, 0)
qgate(CX, 0, 1)
raw = qshow(noise=noise, shots=4096)

# Apply calibration
corrected = cal.apply(raw.counts, 4096)
print(f"Raw:       {raw.counts}")
print(f"Calibrated: {corrected}")

## Stacking ZNE + Readout Calibration

For best results, combine both techniques.

In [ ]:
# Stacked: ZNE + readout calibration
cal = calibrate(2, backend="native", shots=4096, noise=NoiseModel(readout=0.05))
res_stacked = zne(
    c, noise=0.05, target="1", shots=4096,
    calibration=cal, extrapolation="exponential"
)
print(f"Stacked (ZNE + cal): {res_stacked.extrapolated:.3f}")

## Real Hardware Results (Tuna-17)

| Method | Single-bit (n=2) | Multi-bit (n=4) |
|--------|-----------------|-----------------|
| Raw | 0.936 | 0.706 |
| Readout calibration | 0.982 | 0.788 |
| ZNE exponential | 0.920 | 0.812 |
| Stacked | **0.963** | **0.869** |